# Ablation summary figure — Augmentation & Cropping

**Thesis:** *Concept dataset extraction for Concept Activation Vectors: A study on open-vocabulary grounding and segmentation*  
**Author:** Alessandro Cogollo (233022)  
**Advisor:** Prof. Marco Brambilla — **Co-advisors:** A. De Santis, M. Bianchi, R. Campi  
**Academic year:** 2025-2026

**Maps to:** editorial summary of Thesis §4.3.2 + §4.3.3 (companion to **Figure 19**).

## Purpose
Renders a single two-row summary figure of the two construction-choice ablations (augmentation on top, cropping policy on the bottom). **All values are read from the CSVs exported by `ablations-riassuntive` — nothing is hard-coded.**

## Prerequisite
Run `ablations-riassuntive` first; it writes `summary_metrics_cells.csv` and `summary_metrics_layers.csv`. Point `SUMMARY_DIR` at that folder.

## Changelog vs. the previous draft
- ❌ Removed the **TCAV Spearman 0.366→0.93** call-out: no experiment in the thesis or notebooks produces it (TCAV-score consistency is future work, Thesis §5.3).
- ❌ Removed the **hard-coded augmented-CAV placeholders**; the augmented bars now come from the real per-layer CSV.
- ✅ Figure now fails loudly if the CSVs are missing, instead of drawing fictional numbers.

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ACG4CAV — Ablation summary figure (augmentation + cropping policy).

Data source: the two long-format CSVs exported by `ablations-riassuntive`:
    summary_metrics_cells.csv   (source, crop_method, concept, vlm_align_mean,
                                 vlm_align_std, vlm_prec_05, vlm_prec_07,
                                 rep_rate, participation_ratio, cav_cosine, n_images)
    summary_metrics_layers.csv  (source, crop_method, concept, layer, cav_cosine)

No value in this figure is hard-coded; everything is aggregated from those files.
"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ── Where the CSVs live (output of `ablations-riassuntive`) ───────────────────
SUMMARY_DIR = "/kaggle/working/exp_summary_output"
CELLS_CSV  = os.path.join(SUMMARY_DIR, "summary_metrics_cells.csv")
LAYERS_CSV = os.path.join(SUMMARY_DIR, "summary_metrics_layers.csv")

for p in (CELLS_CSV, LAYERS_CSV):
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"Missing {p}. Run `ablations-riassuntive` first and set SUMMARY_DIR "
            f"to its output folder. This figure intentionally does not fabricate data.")

cells  = pd.read_csv(CELLS_CSV)
layers = pd.read_csv(LAYERS_CSV)

SRC_ORDER  = ["vanilla", "augmented"]
CROP_ORDER = ["bbox", "center_mask", "largest_bbox", "sliding_window"]
LAYER_ORDER = ["layer1.2", "layer2.3", "layer3.5", "layer4.2"]

def by_source(col):
    g = cells.groupby("source")[col].mean()
    return {s: float(g.get(s, np.nan)) for s in SRC_ORDER}

def by_crop(col):
    g = cells.groupby(["crop_method", "source"])[col].mean()
    return {s: [float(g.get((m, s), np.nan)) for m in CROP_ORDER] for s in SRC_ORDER}

def cav_by_layer():
    g = layers.groupby(["source", "layer"])["cav_cosine"].mean()
    return {s: [float(g.get((s, l), np.nan)) for l in LAYER_ORDER] for s in SRC_ORDER}

# ── Style ─────────────────────────────────────────────────────────────────────
C_VAN, C_AUG, GRID = "#3a7ca5", "#e07a23", "#cfd4d9"
mpl.rcParams.update({
    "font.size": 10.5, "font.family": "DejaVu Sans",
    "axes.edgecolor": "#5a5f63", "axes.linewidth": 0.9,
    "axes.titlesize": 11.5, "axes.titleweight": "bold", "axes.labelsize": 10,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7, "grid.alpha": 0.7,
})

def style_axis(ax, ymax=None, ylabel=None):
    ax.set_axisbelow(True); ax.grid(axis="y"); ax.grid(axis="x", visible=False)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    if ymax is not None:
        ax.set_ylim(0, ymax)
    if ylabel:
        ax.set_ylabel(ylabel)

def grouped_bars(ax, labels, van, aug, width=0.38, fmt="{:.2f}", rot=0):
    x = np.arange(len(labels))
    b1 = ax.bar(x - width/2, van, width, label="Vanilla", color=C_VAN,
                edgecolor="white", linewidth=0.6)
    b2 = ax.bar(x + width/2, aug, width, label="Augmented", color=C_AUG,
                edgecolor="white", linewidth=0.6)
    for bars, vals in ((b1, van), (b2, aug)):
        for r, v in zip(bars, vals):
            if not np.isnan(v):
                ax.annotate(fmt.format(v), (r.get_x()+r.get_width()/2, r.get_height()),
                            xytext=(0, 2), textcoords="offset points",
                            ha="center", va="bottom", fontsize=8.2, color="#333")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=rot, ha="center" if rot == 0 else "right")
    return b1, b2

# ── Pull the real numbers ─────────────────────────────────────────────────────
align   = by_source("vlm_align_mean")
prec05  = by_source("vlm_prec_05")
prec07  = by_source("vlm_prec_07")
pr_src  = by_source("participation_ratio")
n_src   = by_source("n_images")
crop_align  = by_crop("vlm_align_mean")
crop_prec07 = by_crop("vlm_prec_07")
crop_pr     = by_crop("participation_ratio")
cav_layer   = cav_by_layer()

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15.5, 9.2))
gs = fig.add_gridspec(2, 3, hspace=0.46, wspace=0.26,
                      left=0.06, right=0.985, top=0.86, bottom=0.075)
fig.suptitle("ACG4CAV — Ablation summary: dataset-construction choices",
             fontsize=16, fontweight="bold", y=0.975)
fig.text(0.5, 0.928, "Augmentation effect (top) and cropping strategy (bottom) — "
         "all values aggregated from the exported CSVs",
         ha="center", fontsize=10.5, color="#555")
fig.text(0.012, 0.835, "A · Augmentation  vs  Vanilla", rotation=90, va="center",
         ha="center", fontsize=11, fontweight="bold", color=C_AUG)
fig.text(0.012, 0.36, "B · Cropping strategy", rotation=90, va="center",
         ha="center", fontsize=11, fontweight="bold", color=C_VAN)

# (A1) intrinsic quality
axA = fig.add_subplot(gs[0, 0])
labelsA = ["VLM align\n(×100)", "Prec @ 0.5", "Prec @ 0.7"]
vanA = [align["vanilla"]*100, prec05["vanilla"], prec07["vanilla"]]
augA = [align["augmented"]*100, prec05["augmented"], prec07["augmented"]]
grouped_bars(axA, labelsA, vanA, augA, fmt="{:.1f}")
style_axis(axA, ymax=100, ylabel="score (%)")
axA.set_title("Intrinsic quality")
axA.legend(loc="upper right", frameon=False, fontsize=9)

# (A2) diversity & coverage (normalised to vanilla = 1.0)
axB = fig.add_subplot(gs[0, 1])
x = np.arange(2); w = 0.38
van_raw = [pr_src["vanilla"], n_src["vanilla"]]
aug_raw = [pr_src["augmented"], n_src["augmented"]]
norm_van = [1.0, 1.0]
norm_aug = [aug_raw[0]/van_raw[0], aug_raw[1]/van_raw[1]]
b1 = axB.bar(x - w/2, norm_van, w, color=C_VAN, edgecolor="white", linewidth=0.6,
             label="Vanilla")
b2 = axB.bar(x + w/2, norm_aug, w, color=C_AUG, edgecolor="white", linewidth=0.6,
             label="Augmented")
axB.axhline(1.0, color="#999", lw=0.8, ls="--")
for bars, raw, norm in ((b1, van_raw, norm_van), (b2, aug_raw, norm_aug)):
    for r, rw, nm in zip(bars, raw, norm):
        axB.annotate(f"{rw:.1f}", (r.get_x()+r.get_width()/2, nm),
                     xytext=(0, 2), textcoords="offset points",
                     ha="center", va="bottom", fontsize=8.2, color="#333")
axB.set_xticks(x)
axB.set_xticklabels(["Participation\nratio", "Samples\nper concept (N)"])
style_axis(axB, ymax=1.85, ylabel="relative to vanilla (=1.0)")
axB.set_title("Diversity & coverage")
axB.legend(loc="upper left", frameon=False, fontsize=9)

# (A3) CAV cosine per layer (no TCAV box, real augmented bars)
axC = fig.add_subplot(gs[0, 2])
grouped_bars(axC, LAYER_ORDER, cav_layer["vanilla"], cav_layer["augmented"],
             fmt="{:.2f}", rot=20)
style_axis(axC, ymax=1.05, ylabel="cosine sim. (auto ↔ manual)")
axC.set_title("CAV similarity per layer")
axC.legend(loc="upper left", frameon=False, fontsize=9)

# (B1) cropping — alignment
axD = fig.add_subplot(gs[1, 0])
grouped_bars(axD, CROP_ORDER, crop_align["vanilla"], crop_align["augmented"],
             fmt="{:.2f}", rot=20)
style_axis(axD, ymax=0.95, ylabel="VLM alignment (mean)")
axD.set_title("VLM alignment per crop method")
axD.legend(loc="upper right", frameon=False, fontsize=9)

# (B2) cropping — precision@0.7
axE = fig.add_subplot(gs[1, 1])
grouped_bars(axE, CROP_ORDER, crop_prec07["vanilla"], crop_prec07["augmented"],
             fmt="{:.0f}", rot=20)
style_axis(axE, ymax=100, ylabel="precision @ 0.7 (%)")
axE.set_title("Strict precision per crop method")
axE.legend(loc="upper right", frameon=False, fontsize=9)

# (B3) cropping — participation ratio
axF = fig.add_subplot(gs[1, 2])
grouped_bars(axF, CROP_ORDER, crop_pr["vanilla"], crop_pr["augmented"],
             fmt="{:.1f}", rot=20)
style_axis(axF, ymax=None, ylabel="participation ratio")
axF.set_title("Diversity per crop method")
axF.legend(loc="upper left", frameon=False, fontsize=9)

fig.text(0.06, 0.012,
         "Vanilla = raw extractions · Augmented = augmentation pipeline.  "
         "Backbone: ResNet-50.  CAV estimator: difference-of-means in standardized space.  "
         "Source: summary_metrics_cells.csv / summary_metrics_layers.csv.",
         fontsize=8, color="#777")

out_png = os.path.join(SUMMARY_DIR, "ablation_summary.png")
out_pdf = os.path.join(SUMMARY_DIR, "ablation_summary.pdf")
fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(out_pdf, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved:", out_png, "/", out_pdf)


FileNotFoundError: Missing /kaggle/working/exp_summary_output/summary_metrics_cells.csv. Run `ablations-riassuntive` first and set SUMMARY_DIR to its output folder. This figure intentionally does not fabricate data.